# Class 5 (workshop version): Can a Language Model Do Our Jobs?

In **Class 1** you trained a classifier. In **Class 2** you fitted a line. Both needed
labelled data and a training step.

Today we hand the same jobs to a language model and **measure** it.

| Part | Job | Data | New idea |
|---|---|---|---|
| 1 | Sentiment of customer reviews | 8 reviews (+ your own) | **zero-shot** |
| 2 | Which patients have the disease | Class 1's screening study | — |
| 3 | What is this house worth | Class 2's ten houses | **few-shot** |
| 4 | What it all cost | the bill | — |

**Zero-shot** means you just ask — no examples, no training. That is Parts 1 and 2.
In Part 3 we add the second lever: **few-shot**, where you paste a few solved examples into
the question first.

> ▶︎ Run cells with **Shift + Enter**, top to bottom. Cells marked ✏️ are **yours to edit**.
> There are thirteen of them.

In [18]:
# If running on Google Colab, clone the repo (if needed),
# move into the repo directory, and ensure it’s on the Python path.

import sys, os

def in_colab():
    try: import google.colab; return True
    except: return False

if in_colab():
    repo = "Hands-On-Notebooks"
    if os.path.basename(os.getcwd()) != repo:
        if not os.path.exists(repo):
            !git clone https://github.com/BridgingAISocietySummerSchools/{repo}
        %cd {repo}
    if '.' not in sys.path:
        sys.path.append('.')

In [19]:
# Setup — run once.

import numpy as np
import pandas as pd

from plotting_utils.llm_simple import (
    REVIEWS, HOUSE_SIZES, HOUSE_PRICES,
    to_label, to_number, to_price_in_thousands,
    show_label_results, plot_accuracy_bars, create_classifier_playground,
)
from plotting_utils.llm_benchmark import (
    run_in_parallel, load_screening_benchmark, patient_to_text,
    score_classification, show_scoreboard, plot_metric_comparison,
    score_regression, plot_price_comparison, plot_mae_bars, cost_projection,
)

print("✅ Ready.")

✅ Ready.


### 🔑 The key

Your instructor will give you one. It goes in the **hidden box** below — never in a cell.
(On Colab you can instead add `OPENROUTER_API_KEY` under 🔑 **Secrets**, with *Notebook
access* switched on.)

In [20]:
import os, getpass
from llm_client import ask_llm, llm_available, describe_setup, print_usage, USAGE

if not llm_available():
    try:
        key = getpass.getpass("Paste the workshop key (hidden): ").strip()
    except Exception:
        key = ""
    if key:
        os.environ["OPENROUTER_API_KEY"] = key

# A small, fast model: under a second per answer, and cheap. Part 4 explains why.
os.environ["OPENROUTER_MODEL"] = "anthropic/claude-haiku-4.5"

LLM_READY = describe_setup()

🔌 Connected. Model: anthropic/claude-haiku-4.5
   Key loaded from a .env file or the environment — never from this notebook.


In [21]:
# ⏱️ Sizes. Small on purpose, so every cell finishes while you are still looking at it.

N_PATIENTS = 20      # patients we send to the model in Part 2
EX_PATIENTS = 10     # a smaller sample again, for the ✏️ exercises

---
# Part 1 · Sentiment ⏱️ 30 min

The job: read a customer review, say **positive** or **negative**.

Class 1's way: collect thousands of labelled reviews, train, evaluate.
Today's way: **ask**. That is all zero-shot means.

In [22]:
def ask(question):
    """Text in, text out. This is the whole interface."""
    return ask_llm(question, max_tokens=400)


print(ask("In two sentences, what is machine learning?"))

Machine learning is a type of artificial intelligence where computers learn patterns from data and improve their performance on tasks without being explicitly programmed for each scenario. It works by identifying patterns in training data and using those patterns to make predictions or decisions on new, unseen data.


### ✏️ Exercise 1 — Ask it something you can check (3 min)

Replace the question with one **from your own field, where you already know the answer**.
Then judge it. Was it right? Was it *confidently* wrong? Could you tell the difference from
the answer alone?

In [23]:
# ✏️ EXERCISE 1 — change the text in the quotes.

print(ask("Explain what a p-value is to someone who has never studied statistics."))

# P-Value Explained Simply

Imagine you flip a coin 10 times and get heads 8 times. You might wonder: "Is this coin fair, or is it rigged?"

A **p-value answers: "If the coin were actually fair, how likely is it I'd see results this extreme (or more extreme) just by random chance?"**

## A Concrete Example

Let's say the p-value comes back as **0.03** (or 3%).

This means: *If the coin were fair, there's only a 3% chance you'd randomly get 8+ heads in 10 flips.*

Since 3% is pretty unlikely, you'd probably conclude the coin is rigged.

## The Key Insight

- **High p-value** (like 0.50) = Your results are totally normal for a fair coin. No reason to suspect anything.
- **Low p-value** (like 0.03) = Your results are surprising if the coin is fair. Maybe something else is going on.

## Important Caveat

A low p-value doesn't *prove* the coin is rigged—it just means the evidence points that way. There's still that small chance you just got unlucky.

**In short:** P-value = "How surprised s

### 🏷️ A classifier in six lines

Two things are doing the work:

- **"Answer with the category name only."** — otherwise you get a paragraph, and a paragraph
  is not a label.
- **`to_label(...)`** — turns *"Clearly positive."* back into `positive`.

In [24]:
def classify(text, labels):
    """Sort `text` into exactly one of `labels`. Zero-shot: no examples, no training."""
    prompt = (f"Classify the text into exactly one of these categories: {', '.join(labels)}.\n"
              f"Answer with the category name only.\n\n"
              f"Text: {text}")
    return to_label(ask_llm(prompt, max_tokens=20), labels)


print(classify("The battery lasts all day and the screen is gorgeous.", ["positive", "negative"]))

positive


### ✏️ Exercise 2 — Predict, *then* run (5 min)

Write three reviews you think are **hard**. Sarcasm and faint praise are your best weapons.

**Before you run the cell:** say out loud what you expect for each one. The interesting
reviews are the ones where you and the model disagree.

In [25]:
# ✏️ EXERCISE 2 — edit these three lines. Predict first, then run.

my_tricky = [
    "Well, it arrived. Eventually.",
    "I have owned worse toasters.",
    "The colour is nice.",
]

for text in my_tricky:
    print(f"{classify(text, ['positive', 'negative']):>8}  |  {text}")

negative  |  Well, it arrived. Eventually.
positive  |  I have owned worse toasters.
positive  |  The colour is nice.


### ✏️ Exercise 3 — Take the format instruction away (4 min)

The line *"Answer with the category name only"* looks like politeness. It is not. Below is
the same job asked casually, the way you would ask a colleague.

Run it and read the **raw replies**. Then look at what `to_label` managed to extract.

> 📌 This is the tax you pay for a model that speaks English: **a trained model hands you a
> number; a language model hands you prose, and prose has to be parsed.** Some of it will not
> parse, and a real system needs a plan for those — not an exception.

In [26]:
# ✏️ EXERCISE 3 — run as-is first. Then try to fix the prompt so all three parse.

def classify_casually(text):
    reply = ask_llm(f"Is this customer review positive or negative?\n\n{text}", max_tokens=120)
    print(f"RAW REPLY: {reply[:90]}...")
    print(f"  → to_label() extracted: {to_label(reply, ['positive', 'negative'])}\n")


for r in REVIEWS[5:8]:
    classify_casually(r["text"])

RAW REPLY: This review is **negative**.

The sarcasm in "Oh it's wonderful" combined with the complai...
  → to_label() extracted: negative

RAW REPLY: This review is **mixed/negative overall**.

**Positive aspect:** Excellent sound quality

...
  → to_label() extracted: unclear

RAW REPLY: This review is **mixed/positive overall**.

**Negative aspect:** The shipping was slow ("p...
  → to_label() extracted: unclear



### 📏 Measuring it properly

Eight reviews with labels **we agreed in advance**. The model never sees the labels — they
exist only so we can grade it, exactly like Class 1's test set.

> 📌 We still need labelled data. Not to **train** the model — to **trust** it.

And it does not get to compete against nothing. Two baselines:

- **"Always positive"** — no reading required.
- **Keyword counting** — nice words vs nasty words. No AI, free, instant.

In [27]:
for r in REVIEWS:
    print(f"{r['label']:>8}  |  {r['text']}")

positive  |  The battery lasts all day and the screen is gorgeous. Best purchase this year.
negative  |  Arrived broken, and support never replied to any of my three emails.
positive  |  Does exactly what it promises. No complaints at all.
negative  |  Cheap plastic. It stopped working after two weeks.
positive  |  Setup took ten minutes and it has worked flawlessly ever since.
negative  |  Oh it's wonderful, if you enjoy reading a 60-page manual to turn on a lamp.
negative  |  The sound quality is excellent, but the app crashes every single day.
positive  |  Shipping was painfully slow, but honestly the product is worth the wait.


In [28]:
GOOD = ["great", "gorgeous", "excellent", "flawlessly", "best", "worth", "nice", "wonderful"]
BAD = ["broken", "cheap", "stopped", "crashes", "slow", "never"]

def keyword_rule(text):
    """No AI at all: count nice words vs nasty words."""
    t = text.lower()
    return "positive" if sum(w in t for w in GOOD) > sum(w in t for w in BAD) else "negative"


truth = [r["label"] for r in REVIEWS]
scores = {
    "Always 'positive'": sum(t == "positive" for t in truth) / len(truth),
    "Keyword counting": sum(keyword_rule(r["text"]) == t for r, t in zip(REVIEWS, truth)) / len(truth),
}

if LLM_READY:
    guesses = run_in_parallel(lambda r: classify(r["text"], ["positive", "negative"]), REVIEWS)
    scores["Language model"] = show_label_results([r["text"] for r in REVIEWS], truth, guesses)

plot_accuracy_bars(scores, title="Sentiment on the same 8 reviews")

········  (8 calls)


Review,Our label,Model said,
The battery lasts all day and the screen is gorgeous. Best …,positive,positive,✅
"Arrived broken, and support never replied to any of my thre…",negative,negative,✅
Does exactly what it promises. No complaints at all.,positive,positive,✅
Cheap plastic. It stopped working after two weeks.,negative,negative,✅
Setup took ten minutes and it has worked flawlessly ever si…,positive,positive,✅
"Oh it's wonderful, if you enjoy reading a 60-page manual to…",negative,negative,✅
"The sound quality is excellent, but the app crashes every s…",negative,negative,✅
"Shipping was painfully slow, but honestly the product is wo…",positive,positive,✅



🎯 The model agreed with us on 8 of 8 reviews (100%) — with zero training examples.


### 💬 Two things to notice

1. **The last three reviews are the whole test.** Sarcasm, *good-then-bad*, *bad-then-good*.
   Keyword counting cannot ever get those; the model reads them the way you do.
2. **Our labels are opinions.** We *decided* "excellent sound, crashes daily" is negative. When
   the model disagrees with a label, ask first whether the label was right.

If the model scored 8/8, **the test is too easy to learn anything more from it.** So build a
harder one. That is the next exercise, and it is the single most useful habit here.

### ✏️ Exercise 4 — Build a harder test set (7 min)

Write **four reviews with the label you believe is correct**. Aim for cases you had to think
about.

The four already in the cell are **faint praise**: every word is positive, the verdict is not.
Replace them with your own — mixed praise and complaint, sarcasm, a review in your second
language, one that is genuinely neutral, a five-word review.

> ⚠️ Two of the four are arguable. Is *"Exactly what I expected."* really negative? Hold on to
> that when the model "gets it wrong".

In [29]:
# ✏️ EXERCISE 4 — write four reviews and YOUR label for each.

MY_REVIEWS = [
    {"text": "Exactly what I expected.",                                       "label": "negative"},
    {"text": "Five stars for the packaging.",                                  "label": "negative"},
    {"text": "Not the prettiest, but three years in and it has never failed.", "label": "positive"},
    {"text": "Works as advertised, which these days is a compliment.",         "label": "positive"},
]

# The three hard originals plus yours -- our test set from here on.
HARD = REVIEWS[5:] + MY_REVIEWS

if LLM_READY:
    hard_truth = [r["label"] for r in HARD]
    hard_guesses = run_in_parallel(lambda r: classify(r["text"], ["positive", "negative"]), HARD)
    zero_shot_hard = show_label_results([r["text"] for r in HARD], hard_truth, hard_guesses)

·······  (7 calls)


Review,Our label,Model said,
"Oh it's wonderful, if you enjoy reading a 60-page manual to…",negative,negative,✅
"The sound quality is excellent, but the app crashes every s…",negative,negative,✅
"Shipping was painfully slow, but honestly the product is wo…",positive,positive,✅
Exactly what I expected.,negative,positive,❌
Five stars for the packaging.,negative,positive,❌
"Not the prettiest, but three years in and it has never fail…",positive,positive,✅
"Works as advertised, which these days is a compliment.",positive,positive,✅



🎯 The model agreed with us on 5 of 7 reviews (71%) — with zero training examples.


### ✏️ Exercise 5 — Fix it with the prompt (7 min)

You cannot retrain this model. You **can** rewrite the question — and that is the entire
engineering discipline you get with a zero-shot model.

Add **one line** where marked, then re-grade on the same hard set.

Things people try, roughly in order of how well they work:

- `"Look for faint praise and implied criticism: would the writer recommend it?"`
- `"Sarcasm is common. Judge the writer's overall verdict, not individual words."`
- `"If a review mentions both good and bad, the deciding factor is whether they would buy again."`

In [30]:
# ✏️ EXERCISE 5 — add or change ONE line where marked.

def my_classify(text, labels):
    prompt = (f"Classify the text into exactly one of these categories: {', '.join(labels)}.\n"
              # ✏️ ADD YOUR LINE HERE -------------------------------------------------
              ""
              # -----------------------------------------------------------------------
              f"Answer with the category name only.\n\n"
              f"Text: {text}")
    return to_label(ask_llm(prompt, max_tokens=20), labels)


if LLM_READY:
    my_guesses = run_in_parallel(lambda r: my_classify(r["text"], ["positive", "negative"]), HARD)
    my_prompt_hard = show_label_results([r["text"] for r in HARD], hard_truth, my_guesses,
                                        note="— with my extra instruction.")

    plot_accuracy_bars({"Just asking": zero_shot_hard, "My instruction": my_prompt_hard},
                       title=f"On the same {len(HARD)} hard reviews")

·······  (7 calls)


Review,Our label,Model said,
"Oh it's wonderful, if you enjoy reading a 60-page manual to…",negative,negative,✅
"The sound quality is excellent, but the app crashes every s…",negative,negative,✅
"Shipping was painfully slow, but honestly the product is wo…",positive,positive,✅
Exactly what I expected.,negative,positive,❌
Five stars for the packaging.,negative,positive,❌
"Not the prettiest, but three years in and it has never fail…",positive,positive,✅
"Works as advertised, which these days is a compliment.",positive,positive,✅



🎯 The model agreed with us on 5 of 7 reviews (71%) — with my extra instruction.


### 💬 Did your line help?

Maybe. On seven reviews, **one review is 14 percentage points** — so unless the bar moved a
long way, you have measured noise, not an improvement. Nobody can tell two prompts apart on a
test set this small, and that includes the people selling you models.

Two honest outcomes, both worth having:

- **It helped** → you fixed a real blind spot by writing one sentence. No retraining, no data.
- **It did not** → the case may be genuinely ambiguous, and no instruction will settle what
  humans disagree about.

### ✏️ Exercise 6 — The thing a trained model cannot do (5 min)

Class 1's classifier answers exactly one question. To make it sort support emails instead you
would need a new labelled data set and a new training run: **days**.

Here you change a sentence. Type any text and any categories, press **Run**.

Try: `urgent, normal` · `billing, technical, other` · `happy, angry, confused` ·
or categories from your own work.

In [32]:
create_classifier_playground(classify)

💡 Change either box, then press 'Run'. Some category sets to try:
   • urgent, normal        • billing, technical, other
   • happy, angry, confused, neutral



interactive(children=(Textarea(value='My train was 40 minutes late again.', description='Text:', layout=Layout…

---
# Part 2 · The Patients from Class 1 ⏱️ 25 min

Same job as Class 1: **does this patient have the disease?** Same data, same held-out
patients, same trained model — and the same zero-shot trick as Part 1.

| | Sees the training data? |
|---|---|
| **Class 1's trained model** | all 7,000 patients, during training |
| **The language model** | **nothing.** It has never seen this study |

> ⚖️ Two setup notes. Our sample is **half sick, half healthy** so that "how many did it
> catch?" is measurable at all — so "always say healthy" scores 50% here, not 90%. And because
> Class 1's model was trained where only 10% are ill, we ask it to flag anyone above **10%**
> risk rather than 50%. Same model, fair question.

In [33]:
bench = load_screening_benchmark(n_benchmark=N_PATIENTS)
bench.describe()

📋 Class 1's study: 10,000 patients, 10.0% of them with the disease.
   Trained on 7,000, held out 3,000.
   Class 1's model on the full held-out set: accuracy 92.6%, AUC 0.887

🎯 Our benchmark: 20 of those held-out patients (10 with the disease, 10 without).
   Deliberately balanced, so 'always say healthy' scores 50% here, not 90%.
   Every approach in this notebook is judged on these same 20 people.
   Because of that, Class 1's model also gets its threshold moved from 0.50 to 0.10, to match this sample.


/Users/christophweisser/Desktop/Coding/Hands-On-Notebooks/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning:

divide by zero encountered in matmul

/Users/christophweisser/Desktop/Coding/Hands-On-Notebooks/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning:

overflow encountered in matmul

/Users/christophweisser/Desktop/Coding/Hands-On-Notebooks/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning:

invalid value encountered in matmul

/Users/christophweisser/Desktop/Coding/Hands-On-Notebooks/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/christophweisser/Desktop/Coding/Hands-On-Notebooks/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/christophweisser/Desktop/Coding/Hands-On-Notebooks/.venv/lib/python3.11/site-pac

### ✍️ A table row is not a question

Class 1's model takes six numbers. A language model takes English — so somebody has to decide
how to *say* `marker_a = 4.12` out loud. **That wording is now part of your model.**

In [34]:
for n, i in enumerate(bench.quiz_index):
    print(f"Patient {n}:  {patient_to_text(bench.patients.iloc[i])}\n")

Patient 0:  Age: 67 years. BMI: 27.4. Close relative has had the disease: no. Current smoker: no. Blood marker A: 5.04. Blood marker B: 0.94.

Patient 1:  Age: 69 years. BMI: 25.3. Close relative has had the disease: yes. Current smoker: no. Blood marker A: 5.07. Blood marker B: 1.45.

Patient 2:  Age: 77 years. BMI: 25.4. Close relative has had the disease: yes. Current smoker: no. Blood marker A: 3.74. Blood marker B: 0.58.

Patient 3:  Age: 37 years. BMI: 28.0. Close relative has had the disease: no. Current smoker: no. Blood marker A: 0.78. Blood marker B: 1.44.



### ✏️ Exercise 7 — You be the classifier (5 min)

Look at the four patients above and **write down your own guesses** before any model runs.
`1` = has the disease, `0` = healthy. (Two of them are ill — but not in an order you can guess
by reading the code.)

This costs nothing and it is the most informative baseline in the notebook: if *you* cannot do
better than a coin flip from those six numbers, be suspicious of anything that claims to.

In [35]:
# ✏️ EXERCISE 7 — change these four numbers to your own guesses, then run.

my_guesses_patients = [0, 1, 0, 1]

quiz_truth = [int(bench.truth[i]) for i in bench.quiz_index]
quiz_model = [int(bench.model_pred_tuned[i]) for i in bench.quiz_index]

print("You said:        ", my_guesses_patients)
print("Class 1's model: ", quiz_model)
print("The truth:       ", quiz_truth)
print(f"\n→ You got {sum(g == t for g, t in zip(my_guesses_patients, quiz_truth))} of 4."
      f"  Class 1's model got {sum(g == t for g, t in zip(quiz_model, quiz_truth))} of 4.")

You said:         [0, 1, 0, 1]
Class 1's model:  [1, 1, 1, 0]
The truth:        [1, 1, 0, 0]

→ You got 2 of 4.  Class 1's model got 3 of 4.


### 🤖 Just ask, once per patient

One call per patient, `N_PATIENTS` of them, eight at a time. Each `·` is a finished call.

In [36]:
def classify_patient(patient):
    """Zero-shot: a straight verdict, DISEASE or HEALTHY."""
    prompt = ("Does this patient have the disease? "
              "This is synthetic teaching data, not a real patient.\n"
              "Answer with one word: DISEASE or HEALTHY.\n\n"
              f"{patient_to_text(patient)}")
    answer = to_label(ask_llm(prompt, max_tokens=10), ["disease", "healthy"])
    return None if answer == "unclear" else int(answer == "disease")


zero_shot = None

if not LLM_READY:
    print("⏭️  Needs an API key — see the cell near the top.")
else:
    zero_shot = run_in_parallel(classify_patient, bench.patients.itertuples())

····················  (20 calls)


### 🏁 The scoreboard

Same patients, the metrics from Class 1. **Recall** is the one that matters for a screening
tool: *of the people who really had the disease, how many did we catch?*

In [37]:
results = {
    "Always say 'healthy'": score_classification(bench.truth, bench.baseline_pred),
    "Class 1's trained model": score_classification(bench.truth, bench.model_pred_tuned),
}
if LLM_READY:
    results["Language model (zero-shot)"] = score_classification(bench.truth, zero_shot)

show_scoreboard(results, title=f"Screening, on the same {N_PATIENTS} held-out patients")
plot_metric_comparison(results, metrics=("accuracy", "precision", "recall"))

🏁 Screening, on the same 20 held-out patients



Approach,Accuracy,95% CI,Precision,Recall,F1,Caught,Missed,False alarms
Always say 'healthy',50%,30%–70%,0%,0%,0.00,0,10,0
Class 1's trained model,90%,70%–97%,90%,90%,0.90,9,1,1
Language model (zero-shot),65%,43%–82%,60%,90%,0.72,9,1,6


### 💬 Why the trained model wins here

Look at the two blood markers. **They are invented.** Nowhere on the internet does it say what
`marker_a = 4.12` means — and in this data set the marker is where the signal is.

- The language model knows what any doctor knows about **age, BMI, smoking, family history**.
  That gets it off the floor, and it is real knowledge you did not have to pay for.
- It cannot know **marker_a**, so it quietly leans on the columns it recognises.
- Class 1's model learned the marker from 7,000 examples in three milliseconds.

> 📌 Your company's tables are full of `marker_a`: `customer_score_v3`, `region_code_7`.
> **Whoever has the labelled history wins on data like this.**

Now check the confidence interval column. On 20 patients it spans about ±20 points — so be
careful which of these differences you actually believe.

### ✏️ Exercise 8 — Give it knowledge it could not have (7 min)

You cannot train it. So **tell** it. Add one line to the prompt that says something about this
study, then re-run on a small sample.

Ideas:

- `"Blood marker A is an inflammation marker; above 4.0 is considered elevated."`
- `"About 10% of patients in this study have the disease."`
- `"The marker matters most in patients over 60."`  ← this one happens to be true here

**The question to answer:** if inventing a meaning for `marker_a` improves the score, what
have you learned — about the model, or about yourself?

In [38]:
# ✏️ EXERCISE 8 — add your line where marked, then run.

def my_classify_patient(patient):
    prompt = ("Does this patient have the disease? This is synthetic teaching data.\n"
              # ✏️ ADD YOUR LINE HERE -------------------------------------------------
              ""
              # -----------------------------------------------------------------------
              "Answer with one word: DISEASE or HEALTHY.\n\n"
              f"{patient_to_text(patient)}")
    answer = to_label(ask_llm(prompt, max_tokens=10), ["disease", "healthy"])
    return None if answer == "unclear" else int(answer == "disease")


if LLM_READY:
    sample = bench.patients.head(EX_PATIENTS)
    mine = run_in_parallel(my_classify_patient, sample.itertuples())

    show_scoreboard({
        "Just asking": score_classification(bench.truth[:EX_PATIENTS], zero_shot[:EX_PATIENTS]),
        "My extra line": score_classification(bench.truth[:EX_PATIENTS], mine),
        "Class 1's model": score_classification(bench.truth[:EX_PATIENTS],
                                                bench.model_pred_tuned[:EX_PATIENTS]),
    }, title=f"On the first {EX_PATIENTS} patients")

··········  (10 calls)
🏁 On the first 10 patients



Approach,Accuracy,95% CI,Precision,Recall,F1,Caught,Missed,False alarms
Just asking,70%,40%–89%,71%,83%,0.77,5,1,2
My extra line,70%,40%–89%,71%,83%,0.77,5,1,2
Class 1's model,90%,60%–98%,100%,83%,0.91,5,1,0


### ✏️ Exercise 9 — *Which* patients does it get wrong? (7 min, no API calls)

Class 1 spent a whole section on this: an accuracy number tells you *how often*, never *who*.
The cell below lists every patient the language model got wrong.

Start with the question Class 1 would ask first: **which kind of mistake is it making?**

- **False alarms or missed cases?** One means healthy people get called back for tests. The
  other means sick people are sent home. They are not the same thing.
- Does **Class 1's model** get those same patients right, or are they hard for everyone?
- Look down `age` and `marker_a` in the failures — is there a pattern, or are you seeing one
  because you are looking for one? (Seven rows is not many.)

Then finish the sentence at the bottom of the cell.

In [39]:
# ✏️ EXERCISE 9 — run, look, then write your sentence at the bottom.

if LLM_READY:
    check = bench.patients[["age", "bmi", "family_history", "smoker", "marker_a", "marker_b"]].copy()
    check["truth"] = bench.truth
    check["Class 1's model"] = bench.model_pred_tuned
    check["language model"] = [-1 if g is None else g for g in zero_shot]

    wrong = check[check["language model"] != check["truth"]]
    print(f"The language model got {len(wrong)} of {len(check)} wrong "
          f"(1 = disease, 0 = healthy, -1 = unparseable reply):\n")
    display(wrong)

    false_alarms = (wrong["truth"] == 0).sum()
    missed = (wrong["truth"] == 1).sum()
    also_wrong = (wrong["Class 1's model"] != wrong["truth"]).sum()

    print(f"Of its {len(wrong)} mistakes:  "
          f"{false_alarms} false alarm{'s' if false_alarms != 1 else ''}, "
          f"{missed} missed case{'s' if missed != 1 else ''}.")
    print(f"Class 1's model got {len(wrong) - also_wrong} of those same "
          f"{len(wrong)} patients right.\n")
    print(f"Average age — got it wrong: {wrong['age'].mean():.0f}   "
          f"got it right: {check[check['language model'] == check['truth']]['age'].mean():.0f}")

# ✏️ Your sentence:
my_pattern = "It tends to be wrong about patients who are ..."
print("\n" + my_pattern)

The language model got 7 of 20 wrong (1 = disease, 0 = healthy, -1 = unparseable reply):



,age,bmi,family_history,smoker,marker_a,marker_b,truth,Class 1's model,language model
2,67,27.4,0,0,5.04,0.94,1,1,0
4,62,29.5,1,0,2.52,1.02,0,0,1
8,35,30.9,0,1,1.99,1.67,0,0,1
10,26,28.6,1,0,2.22,1.00,0,0,1
12,77,25.4,1,0,3.74,0.58,0,1,1
13,61,31.4,0,0,3.22,1.02,0,0,1
17,58,26.1,1,0,3.77,0.24,0,0,1


Of its 7 mistakes:  6 false alarms, 1 missed case.
Class 1's model got 6 of those same 7 patients right.

Average age — got it wrong: 55   got it right: 60

It tends to be wrong about patients who are ...


---
# Part 3 · The Houses from Class 2 ⏱️ 25 min

Classification asked *which category?* Regression asks **how much?**

Class 2 had ten house sales. We split them **five and five**:

- **5 houses everyone may learn from** — Class 2's line is fitted on these.
- **5 houses nobody sees** — everyone predicts these.

In [40]:
from sklearn.linear_model import LinearRegression

sizes, prices = np.array(HOUSE_SIZES), np.array(HOUSE_PRICES)
shown, hidden = np.array([0, 2, 4, 6, 8]), np.array([1, 3, 5, 7, 9])

print("🏠 The 5 houses everyone gets to see:")
for s, p in zip(sizes[shown], prices[shown]):
    print(f"   {s:,} sq ft sold for ${p}k")

print("\n❓ The 5 houses we have to predict:")
for s in sizes[hidden]:
    print(f"   {s:,} sq ft → ?")

line = LinearRegression().fit(sizes[shown].reshape(-1, 1), prices[shown])
line_guesses = line.predict(sizes[hidden].reshape(-1, 1))
print(f"\n📏 Class 2's line, fitted on those 5: ${line.coef_[0] * 1000:.0f} per square foot.")

🏠 The 5 houses everyone gets to see:
   800 sq ft sold for $150k
   1,200 sq ft sold for $215k
   1,600 sq ft sold for $270k
   2,000 sq ft sold for $350k
   2,400 sq ft sold for $410k

❓ The 5 houses we have to predict:
   1,000 sq ft → ?
   1,400 sq ft → ?
   1,800 sq ft → ?
   2,200 sq ft → ?
   2,600 sq ft → ?

📏 Class 2's line, fitted on those 5: $164 per square foot.


### ✏️ Exercise 10 — Beat the line (5 min)

Look at the five known sales and **write down your own guess** for each hidden house, in
thousands. Then run — you are scored alongside everyone else.

In [41]:
# ✏️ EXERCISE 10 — replace these five numbers with your own guesses.

my_price_guesses = [180, 250, 300, 380, 420]

print("Size      you    real")
for s, g, real in zip(sizes[hidden], my_price_guesses, prices[hidden]):
    print(f"{s:>5,}   {g:>4}   {real:>4}    (off by ${abs(g - real) * 1000:,})")

print(f"\nYour average miss: ${score_regression(prices[hidden], my_price_guesses)['mae'] * 1000:,.0f}")

Size      you    real
1,000    180    185    (off by $5,000)
1,400    250    230    (off by $20,000)
1,800    300    295    (off by $5,000)
2,200    380    415    (off by $35,000)
2,600    420    435    (off by $15,000)

Your average miss: $16,000


### 🤖 Zero-shot, one more time

Exactly the Part 1 move: ask, no examples. The model has never seen our neighbourhood, so it
has to silently pick a country, a decade and a market before it can answer.

In [42]:
def price_zero_shot(size):
    prompt = (f"A house is {size} square feet. Estimate its price in thousands of US dollars.\n"
              f"Answer with just a number, no words, no dollar sign. Example: 250")
    return to_price_in_thousands(ask_llm(prompt, max_tokens=20))


llm_zero = llm_few = None
if LLM_READY:
    llm_zero = run_in_parallel(price_zero_shot, sizes[hidden])
    print("\nIts guesses:", llm_zero)
    print("The truth:  ", [int(p) for p in prices[hidden]])

·····  (5 calls)

Its guesses: [350.0, 280.0, 360.0, 440.0, 390.0]
The truth:   [185, 230, 295, 415, 435]


### 💬 That error is not about houses

It is about **which market it guessed**. If it assumed somewhere like ours it looks brilliant;
if it assumed San Francisco it looks absurd — and neither outcome tells you whether it can do
regression.

The problem is obvious: **it does not know anything about our neighbourhood.** We cannot train
it. But we can *show* it.

## 🎓 The second lever: few-shot

Instead of only describing the task, **paste solved examples into the question**. That is
called **few-shot**, or *in-context learning*, and it is the other half of working with these
models.

Three things to be clear about:

- **This is not training.** The examples are re-sent with every single question, and the model
  forgets them the moment it answers. You are paying for them every time.
- **The model now has exactly what the line has** — the same five sales, for the same five
  hidden houses. Same information, two completely different ways of using it.
- **It is not guaranteed to help.** On this task it will. On Part 2's patients it often makes
  things *worse* — six columns of numbers are much harder to learn from than five house
  prices. The only way to know is to measure, which is what we are doing.

In [43]:
SALES = "\n".join(f"{s:,} sq ft sold for ${p}k" for s, p in zip(sizes[shown], prices[shown]))
print("What we paste into every question:\n")
print(SALES)


def price_few_shot(size):
    """Few-shot: the same five sales the line was fitted on, pasted into the question."""
    prompt = (f"Here are recent house sales in one neighbourhood:\n{SALES}\n\n"
              f"Estimate the price of a {size:,} sq ft house in the same neighbourhood, "
              f"in thousands of dollars.\n"
              f"Answer with just a number, no words, no dollar sign. Example: 250")
    return to_price_in_thousands(ask_llm(prompt, max_tokens=20))


if LLM_READY:
    llm_few = run_in_parallel(price_few_shot, sizes[hidden])
    print("\nWith examples:", llm_few)
    print("Without:      ", llm_zero)
    print("The truth:    ", [int(p) for p in prices[hidden]])

What we paste into every question:

800 sq ft sold for $150k
1,200 sq ft sold for $215k
1,600 sq ft sold for $270k
2,000 sq ft sold for $350k
2,400 sq ft sold for $410k
·····  (5 calls)

With examples: [187.5, 242.0, 310.0, 370.0, 460.0]
Without:       [350.0, 280.0, 360.0, 440.0, 390.0]
The truth:     [185, 230, 295, 415, 435]


In [44]:
series = {"Class 2's line": line_guesses, "Your guesses": my_price_guesses}
if LLM_READY:
    series["LLM, zero-shot"] = llm_zero
    series["LLM, few-shot"] = llm_few

plot_price_comparison(sizes[hidden], prices[hidden], series, title="The 5 hidden houses")

errors = {name: score_regression(prices[hidden], values) for name, values in series.items()}
plot_mae_bars(errors)
for name, e in errors.items():
    print(f"{name:>18}:  average miss ${e['mae'] * 1000:,.0f}")

    Class 2's line:  average miss $16,550
      Your guesses:  average miss $16,000
    LLM, zero-shot:  average miss $70,000
     LLM, few-shot:  average miss $19,900
